# SocialJax Quickstart Tutorial

Welcome to SocialJax! This tutorial will get you up and running with multi-agent reinforcement learning in minutes.

## What is SocialJax?

SocialJax is a JAX-based framework for multi-agent reinforcement learning, featuring:
- **Sequential Social Dilemmas**: Environments that test cooperation vs. competition
- **Multiple Algorithms**: IPPO, MAPPO, VDN, and SVO
- **JAX Acceleration**: Fast training with GPU support
- **Flexible Configuration**: YAML configs and dataclass-based settings

## Installation

```bash
# Using pip
pip install -r requirements.txt
pip install jaxlib==0.4.23+cuda11.cudnn86 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# Set PYTHONPATH
export PYTHONPATH=./socialjax:$PYTHONPATH
```

Let's get started!

## 1. Import SocialJax

In [ ]:
# Import the main module
import socialjax
from socialjax import make, registered_envs

# Check available environments
print("Available environments:")
for env_name in sorted(registered_envs()):
    print(f"  - {env_name}")

## 2. Create an Environment

SocialJax provides several multi-agent environments. Let's create the **Coin Game** environment, a classic sequential social dilemma where agents must balance collecting coins vs. cooperating.

In [ ]:
# Create the Coin Game environment with 5 agents
env = make('coin_game', num_agents=5)

print(f"Environment: Coin Game")
print(f"Number of agents: {env.num_agents}")
print(f"Observation shape: {env.observation_space().shape}")
print(f"Number of actions: {env.action_space().n}")

## 3. Explore the Environment

Let's reset the environment and take a few random steps to understand how it works.

In [ ]:
import jax
import jax.numpy as jnp

# Reset the environment
key = jax.random.PRNGKey(0)
obs, state = env.reset(key)

print(f"Initial observation shape: {obs.shape}")
print(f"State type: {type(state).__name__}")

# Take a random step
key, action_key = jax.random.split(key)
actions = jax.random.randint(action_key, (env.num_agents,), 0, env.action_space().n)

# Step the environment
obs, state, rewards, dones, infos = env.step(state, actions)

print(f"\nAfter one step:")
print(f"Rewards: {rewards}")
print(f"Dones: {dones}")

## 4. Create an Algorithm

SocialJax provides several multi-agent RL algorithms. Let's create an **IPPO** (Independent PPO) agent.

In [ ]:
from socialjax.algorithms import get_algorithm, list_algorithms

# List available algorithms
print("Available algorithms:")
for algo in list_algorithms():
    print(f"  - {algo}")

# Get the IPPO algorithm class
IPPOAlgorithm = get_algorithm('ippo')
print(f"\nUsing: IPPOAlgorithm")

## 5. Create a Trainer

The Trainer class provides a high-level interface for training algorithms.

In [ ]:
from socialjax.training import Trainer

# Create a trainer with IPPO on Coin Game
trainer = Trainer(
    algorithm='ippo',
    env='coin_game',
    num_agents=5,
    config={
        'total_timesteps': 10000,  # Short run for demo
        'num_envs': 4,
        'num_steps': 16,
        'learning_rate': 0.001,
        'gamma': 0.99,
    }
)

print("Trainer created successfully!")
print(f"Algorithm: {trainer.algorithm_name}")
print(f"Environment: {trainer.env_name}")

## 6. Train the Agent

Now let's train our IPPO agent on the Coin Game environment.

In [ ]:
# Train for a short run (this may take a minute)
print("Training IPPO on Coin Game...")
print("(This is a short demo run - increase total_timesteps for real training)\n")

# Note: For a quick demo, we'll train with minimal steps
# Real training typically uses 1M+ steps
metrics = trainer.train(total_timesteps=5000)

print("\nTraining complete!")
print(f"Total timesteps: {metrics.get('total_timesteps', 'N/A')}")
if 'mean_episode_return' in metrics:
    print(f"Mean episode return: {metrics['mean_episode_return']:.2f}")

## 7. Evaluate the Trained Agent

After training, let's evaluate our agent's performance.

In [ ]:
# Evaluate the trained agent
print("Evaluating trained agent...")

eval_metrics = trainer.evaluate(n_episodes=10)

print(f"\nEvaluation Results:")
print(f"  Mean return: {eval_metrics.mean_return:.2f}")
print(f"  Std return: {eval_metrics.std_return:.2f}")
if hasattr(eval_metrics, 'cooperation_rate'):
    print(f"  Cooperation rate: {eval_metrics.cooperation_rate:.2%}")

## 8. Save and Load Checkpoints

You can save and load model checkpoints for later use.

In [ ]:
import os

# Save checkpoint
checkpoint_path = "checkpoints/quickstart_ippo"
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

trainer.save(checkpoint_path)
print(f"Checkpoint saved to: {checkpoint_path}")

# Load checkpoint (demonstration)
new_trainer = Trainer(algorithm='ippo', env='coin_game', num_agents=5)
new_trainer.load(checkpoint_path)
print(f"Checkpoint loaded successfully!")

## 9. Visualize Agent Behavior

You can generate GIFs to visualize your trained agent's behavior.

In [ ]:
from socialjax.evaluation import save_gif

# Run evaluation with frame capture
# Note: This requires a trained agent
print("To generate visualizations, use the CLI tool:")
print("python scripts/visualize.py --checkpoint checkpoints/quickstart_ippo --env coin_game --output output.gif")

## 10. Other Environments

SocialJax provides several environments for studying social dilemmas:

In [ ]:
# List all environments with descriptions
environments = {
    'coin_game': 'Collect coins while avoiding conflict',
    'clean_up': 'Clean pollution to harvest apples',
    'harvest_common_open': 'Harvest apples from common resource',
    'coop_mining': 'Cooperative mining for resources',
    'territory_open': 'Territory control and resource gathering',
    'pd_arena': 'Prisoner\'s Dilemma in spatial arena',
    'mushrooms': 'Harvest mushrooms sustainably',
    'gift': 'Gift-giving and social learning',
}

print("Available Environments:\n")
for name, desc in environments.items():
    print(f"  {name}: {desc}")

## 11. Command-Line Training

For production training, use the CLI scripts:

In [ ]:
# CLI training example
print("Training with CLI:")
print("""\npython scripts/train.py \\
    --algorithm ippo \\
    --env coin_game \\
    --timesteps 1000000 \\
    --num-envs 8 \\
    --wandb-project socialjax-experiments
""")

print("\nEvaluation with CLI:")
print("""\npython scripts/evaluate.py \\
    --checkpoint checkpoints/ippo_final \\
    --env coin_game \\
    --episodes 50 \\
    --render \\
    --output evaluation.gif
""")

## Summary

In this quickstart, you learned:

1. How to **create environments** with `socialjax.make()`
2. How to **create algorithms** using the registry
3. How to **train agents** with the Trainer class
4. How to **evaluate and save** trained models
5. How to use the **CLI tools** for production training

## Next Steps

- **Tutorial 2**: Creating custom algorithms
- **Tutorial 3**: Creating custom network architectures
- **Tutorial 4**: Using callbacks for logging and monitoring
- **Tutorial 5**: Advanced configuration options

## Resources

- [API Reference](../docs/api_reference.md)
- [Migration Guide](../docs/migration_guide.md)
- [GitHub Repository](https://github.com/your-org/socialjax)